In [2]:
import pandas as pd
import numpy as np

# -----------------------------
# LOAD DATA
# -----------------------------

students_df = pd.read_csv("../data/students.csv")
assignments_df = pd.read_csv("../data/assignments.csv")
courses_df = pd.read_csv("../data/courses.csv")

# -----------------------------
# MERGE ASSIGNMENTS WITH COURSES
# So we know which YearLevel each assignment belongs to
# -----------------------------

assignments_with_courses = assignments_df.merge(
    courses_df[["CourseID", "YearLevel"]],
    on="CourseID",
    how="left"
)

# -----------------------------
# SETTINGS
# -----------------------------

np.random.seed(42)

submissions = []
submission_id = 1

# -----------------------------
# CREATE SUBMISSIONS
# -----------------------------

for _, assignment in assignments_with_courses.iterrows():

    assignment_id = assignment["AssignmentID"]
    due_date = pd.to_datetime(assignment["DueDate"])
    year_level = assignment["YearLevel"]

    # only students in the matching year level should submit this assignment
    matching_students = students_df[students_df["YearLevel"] == year_level]

    for _, student in matching_students.iterrows():

        is_submitted = np.random.choice(
            [True, False],
            p=[0.95, 0.05]
        )

        if is_submitted:
            score = np.random.normal(loc=70, scale=12)
            score = round(np.clip(score, 40, 100), 2)
            submitted_status = "Submitted"
            submission_date = due_date
        else:
            score = None
            submitted_status = "Missing"
            submission_date = None

        submissions.append({
            "SubmissionID": submission_id,
            "AssignmentID": assignment_id,
            "StudentID": student["StudentID"],
            "SubmissionDate": submission_date.date() if submission_date is not None else None,
            "Score": score,
            "SubmittedStatus": submitted_status
        })

        submission_id += 1

# -----------------------------
# CONVERT TO DATAFRAME
# -----------------------------

submissions_df = pd.DataFrame(submissions)

# -----------------------------
# SAVE TO CSV
# -----------------------------

submissions_df.to_csv("../data/submissions.csv", index=False)